# Data Cleaning & Preprocessing - Household Power Consumption

This notebook outlines the step-by-step data cleaning process for the Household Power Consumption dataset.

### Step 1: Import Libraries
We import standard libraries like `pandas` and `numpy` for data manipulation, and visualization tools. 

* **`missingno (msno)`**: A specialized library used to visualize missing values. Instead of just looking at counts, it plots a visual matrix showing exactly where the gaps in the data are, which helps in identifying if missing data is random or follows a pattern.

In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import missingno as msno
import os
import zipfile

zip_file = "household_power_consumption.zip"
txt_file = "household_power_consumption.txt"
if not os.path.exists(txt_file):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Dataset extracted successfully!")

Extracting dataset...
Dataset extracted successfully!


### Step 2: Load the Dataset
We load the text file dataset using semicolons as delimiters.

* **`low_memory=False`**: Normally, pandas processes files in chunks to guess column data types. For large files (~2 million rows) with mixed types, this can trigger warnings or consume excessive memory. Setting it to `False` forces pandas to parse the entire file before determining types, avoiding warnings.

In [2]:
df = pd.read_csv("household_power_consumption.txt", sep=";", low_memory=False)
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


### Step 3: Check Data Info
We inspect the column names, counts of non-null values, and inferred data types.

In [3]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Date                   2075259 non-null  object 
 1   Time                   2075259 non-null  object 
 2   Global_active_power    2075259 non-null  object 
 3   Global_reactive_power  2075259 non-null  object 
 4   Voltage                2075259 non-null  object 
 5   Global_intensity       2075259 non-null  object 
 6   Sub_metering_1         2075259 non-null  object 
 7   Sub_metering_2         2075259 non-null  object 
 8   Sub_metering_3         2049280 non-null  float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


### Step 4: Summary Statistics
We calculate basic statistical measures (mean, min, max, standard deviation) for numerical columns.

In [4]:
df.describe(include=["object","int"])

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2
count,2075259,2075259,2075259,2075259,2075259,2075259,2075259,2075259
unique,1442,1440,4187,533,2838,222,89,82
top,25/11/2010,20:24:00,?,0.000,?,1.000,0.000,0.000
freq,1440,1442,25979,481561,25979,172785,1880175,1436830


### Step 5: Check for Custom Null Placeholders (`?`)
We check each column to see how many entries are recorded as `?`, which is the custom character used in this dataset to represent missing values.

In [5]:
for col in df.columns:
    print(f"{col}: {(df[col] == '?').sum()}")
    

Date: 0
Time: 0
Global_active_power: 25979
Global_reactive_power: 25979
Voltage: 25979
Global_intensity: 25979
Sub_metering_1: 25979
Sub_metering_2: 25979
Sub_metering_3: 0


### Step 6: Replace Custom Nulls with standard NaN
We convert the custom `?` characters to `np.nan` so pandas can recognize them as missing values and handle them using standard functions.

In [6]:
df.replace('?', np.nan, inplace=True)

### Step 7: Re-check Null Counts
We check the sum of missing values per column after replacing custom placeholders.

In [7]:
df.isnull().sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

### Step 8: View Rows with Missing Values
We isolate and view the dates and times for rows containing missing values to see if there is any temporal pattern in the missing records.

In [8]:
missing_rows = df[df.isnull().any(axis=1)]

missing_rows[['Date', 'Time']].head(20)

,Date,Time
6839,21/12/2006,11:23:00
6840,21/12/2006,11:24:00
19724,30/12/2006,10:08:00
19725,30/12/2006,10:09:00
41832,14/1/2007,18:36:00
61909,28/1/2007,17:13:00
98254,22/2/2007,22:58:00
98255,22/2/2007,22:59:00
142588,25/3/2007,17:52:00
190497,28/4/2007,00:21:00


### Step 9: Convert Numeric Columns to Float
We cast all numeric columns (which were loaded as object/string types due to the `?` characters) to proper numeric types.

In [9]:
numeric_cols = df.columns[2:]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

### Step 10: Create DateTime Column
We combine separate `Date` and `Time` string columns into a single pandas `datetime` format so that it can be parsed as a time-series index.

In [10]:
df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")

### Step 11: Set DateTime as Index
We drop the old redundant `Date` and `Time` columns and set the new `DateTime` column as the index of the dataframe.

In [11]:
df.drop(columns=["Date","Time"], inplace=True)
df.set_index("DateTime", inplace=True)

### Step 12: Inspect Head of Processed Data
We inspect the top rows of the modified dataframe to check the index and structure.

In [12]:
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
DateTime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Step 13: Check for Duplicate Indices
We check if there are any duplicate timestamps in our index.

In [13]:
df.index.duplicated().sum()

np.int64(0)

### Step 14: Time-Based Interpolation
We fill the missing values using time-based interpolation.

* **`interpolate(method="time")`**: Instead of filling missing values with static values (like the mean or median), time-based interpolation uses the datetime index to calculate a linear trend between the previous and next available readings. This is a standard and highly accurate method for handling gaps in continuous time-series sensor data.

In [14]:
df.sort_index(inplace=True)
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min')
df = df.reindex(full_idx)

df = df.interpolate(method="time")


### Step 15: Verify No Nulls Remain
We check the sum of missing values once more to verify that the interpolation has successfully filled all data gaps.

In [15]:
df.isnull().sum()

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

### Step 16: Downcast Numerical Columns to Float32
We cast the numeric float columns to float32.

* **`float32` vs `float64`**: By default, pandas loads decimals in double-precision (64-bit float). Since our measurements do not require extreme decimal precision, converting them to single-precision (32-bit float) cuts the memory usage of the dataframe in half without losing critical signal.

In [16]:
for col in df.columns:
    df[col] = df[col].astype("float32")

### Step 17: Check Memory Footprint
We inspect the detailed memory footprint of our dataframe to verify the downcasting results.

In [17]:
df.info(memory_usage='deep')    

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2075259 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Freq: min
Data columns (total 7 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Global_active_power    float32
 1   Global_reactive_power  float32
 2   Voltage                float32
 3   Global_intensity       float32
 4   Sub_metering_1         float32
 5   Sub_metering_2         float32
 6   Sub_metering_3         float32
dtypes: float32(7)
memory usage: 71.2 MB


### Step 18: Save Cleaned Data to Parquet
We save the cleaned and optimized dataset to a Parquet file.

* **`to_parquet`**: Parquet is a compressed, columnar data storage format. It is much more efficient than text files (like CSV), reduces file size significantly, speeds up read/write times for machine learning models, and preserves exact data types and index settings.

In [18]:
df.to_parquet("cleaned_power.parquet")